In [7]:
import torch
from tianshou.data import Collector, VectorReplayBuffer
from tianshou.env import DummyVectorEnv
from tianshou.policy import DQNPolicy
from tianshou.trainer import OffpolicyTrainer
from dengue_envs.envs.dengue_diagnostics import DengueDiagnosticsEnv
from dengue_wrapper import DengueWrapper, CaseByCaseWrapper
from fcn_network import DengueNet
import numpy as np
from typing import Tuple

In [8]:
print(f"CUDA Available: {torch.cuda.is_available()}")

CUDA Available: True


In [9]:
DEVICE = "cuda"

LR = 1e-4
GAMMA = 0.99
N_STEP = 3
TARGET_UPDATE_FREQ = 1000

BUFFER_SIZE = 1000
BATCH_SIZE = 64

EPOCH = 15
STEP_PER_EPOCH = 10000

STEP_PER_COLLECT = 1000
UPDATE_PER_STEP = 0.1

EPS_TRAIN_START = 1.0
EPS_TRAIN_FINAL = 0.05
EPS_TRAIN_DECAY = 50000
EPS_TEST = 0.01
NUM_ENVS = 4
NUM_TEST_ENVS = 4

In [10]:
WORLD_SIZE = 400
MIN_BORDER_DISTANCE = 50
MAX_RADIUS = 100
MIN_RADIUS = 50

def generate_random_center(size: int, margin: int) -> Tuple[int, int]:
    """Gera um par de coordenadas aleatórias dentro dos limites do mapa."""
    x = np.random.randint(margin, size - margin)
    y = np.random.randint(margin, size - margin)
    return int(x), int(y)

In [11]:
def make_env():
    """Função factory para criar o ambiente com randomização espacial e wrappers."""

    dengue_center = generate_random_center(WORLD_SIZE, MIN_BORDER_DISTANCE)
    chik_center = generate_random_center(WORLD_SIZE, MIN_BORDER_DISTANCE)
    dengue_radius = np.random.randint(MIN_RADIUS, MAX_RADIUS)
    chik_radius = np.random.randint(MIN_RADIUS, MAX_RADIUS)

    env = DengueDiagnosticsEnv(
        epilength=60,
        size=WORLD_SIZE,
        clinical_specificity=(0.5, 0.95),
        dengue_center=dengue_center,
        chik_center=chik_center,
        dengue_radius=dengue_radius,
        chik_radius=chik_radius
    )

    env = DengueWrapper(env)
    env = CaseByCaseWrapper(env)

    return env

In [12]:
print(f"VERIFICAÇÃO: O BUFFER_SIZE é {BUFFER_SIZE}")

VERIFICAÇÃO: O BUFFER_SIZE é 1000


In [13]:
if True:
    train_envs = DummyVectorEnv([make_env for _ in range(NUM_ENVS)])
    test_envs = DummyVectorEnv([make_env for _ in range(NUM_TEST_ENVS)])

    env = make_env()
    map_shape = env.observation_space.spaces["map"].shape
    action_shape = env.action_space.n

    net = DengueNet(map_shape, action_shape, device=DEVICE).to(DEVICE)
    optim = torch.optim.Adam(net.parameters(), lr=LR)

    policy = DQNPolicy(
        model=net,
        optim=optim,
        discount_factor=GAMMA,
        estimation_step=N_STEP,
        target_update_freq=TARGET_UPDATE_FREQ,
        action_space=env.action_space
    )

    print(f"DEBUG: TENTANDO USAR BUFFER_SIZE={BUFFER_SIZE}")

    buffer = VectorReplayBuffer(
        total_size=BUFFER_SIZE,
        buffer_num=NUM_ENVS,
        ignore_obs_next=True
    )

    train_collector = Collector(
        policy, train_envs, buffer, exploration_noise=True
    )
    test_collector = Collector(policy, test_envs)

    print("Forçando a coleta inicial de 100 passos para inicialização segura do buffer.")

    train_collector.collect(n_step=100, reset_before_collect=True)

    def train_fn(epoch, env_step):
        if env_step <= EPS_TRAIN_DECAY:
            eps = EPS_TRAIN_START - env_step / EPS_TRAIN_DECAY * \
                  (EPS_TRAIN_START - EPS_TRAIN_FINAL)
        else:
            eps = EPS_TRAIN_FINAL
        policy.set_eps(eps)


    def test_fn(epoch, env_step):
        policy.set_eps(EPS_TEST)

    trainer = OffpolicyTrainer(
        policy=policy,
        train_collector=train_collector,
        test_collector=test_collector,
        max_epoch=EPOCH,
        step_per_epoch=STEP_PER_EPOCH,
        step_per_collect=STEP_PER_COLLECT,
        update_per_step=UPDATE_PER_STEP,
        episode_per_test=NUM_TEST_ENVS,
        batch_size=BATCH_SIZE,
        train_fn=train_fn,
        test_fn=test_fn,
        stop_fn=lambda mean_rewards: mean_rewards >= 100
    )

    print(f"Iniciando treinamento na {DEVICE}...")
    result = trainer.run()
    print("\n--- Resultado do Treinamento ---")
    print(result)

    torch.save(policy.state_dict(), "dqn_dengue_policy4 .pth")
    print("Política salva em dqn_dengue_policy.pth")

DEBUG: TENTANDO USAR BUFFER_SIZE=1000
Forçando a coleta inicial de 100 passos para inicialização segura do buffer.
Reward: 1.0 	 Total Reward: 1.0
Reward: 1.0 	 Total Reward: 1.0
Reward: 1.0 	 Total Reward: 1.0
Reward: 1.0 	 Total Reward: 1.0
Iniciando treinamento na cuda...
Reward: -1.0 	 Total Reward: -1.0
Reward: 1.0 	 Total Reward: 1.0
Reward: 1.0 	 Total Reward: 1.0
Reward: 1.0 	 Total Reward: 1.0
Reward: -1.0 	 Total Reward: -2.0
Reward: 1.0 	 Total Reward: 2.0
Reward: -1.0 	 Total Reward: 0.0
Reward: 7.0 	 Total Reward: 8.0
Reward: 2.0 	 Total Reward: 0.0
Reward: 0.0 	 Total Reward: 2.0
Reward: 2.0 	 Total Reward: 2.0
Reward: 6.0 	 Total Reward: 14.0
Reward: -6.0 	 Total Reward: -6.0
Reward: -10.0 	 Total Reward: -8.0
Reward: 6.0 	 Total Reward: 8.0
Reward: -6.0 	 Total Reward: 8.0
Reward: -1.0 	 Total Reward: -7.0
Reward: -5.0 	 Total Reward: -13.0
Reward: 3.0 	 Total Reward: 11.0
Reward: -5.0 	 Total Reward: 3.0
Reward: -10.0 	 Total Reward: -17.0
Reward: -10.0 	 Total Reward:

Epoch #1:   0%|          | 0/10000 [00:00<?, ?it/s]

Reward: -1.0 	 Total Reward: 0.0
Reward: -10.5 	 Total Reward: -9.5
Reward: 9.5 	 Total Reward: 10.5
Reward: -10.5 	 Total Reward: -9.5
Reward: -88.5 	 Total Reward: -88.5
Reward: -28.0 	 Total Reward: -37.5
Reward: -66.0 	 Total Reward: -55.5
Reward: -97.0 	 Total Reward: -106.5
Reward: -82.0 	 Total Reward: -170.5
Reward: -83.0 	 Total Reward: -120.5
Reward: -98.0 	 Total Reward: -153.5
Reward: -59.0 	 Total Reward: -165.5
Reward: -150.0 	 Total Reward: -320.5
Reward: -148.0 	 Total Reward: -268.5
Reward: -123.0 	 Total Reward: -276.5
Reward: -135.5 	 Total Reward: -301.0
Reward: 16.0 	 Total Reward: -304.5
Reward: -75.0 	 Total Reward: -343.5
Reward: -63.5 	 Total Reward: -340.0
Reward: -43.5 	 Total Reward: -344.5
Reward: -28.0 	 Total Reward: -332.5
Reward: -18.5 	 Total Reward: -362.0
Reward: -52.0 	 Total Reward: -392.0
Reward: -4.5 	 Total Reward: -349.0
Reward: -56.5 	 Total Reward: -389.0
Reward: -80.5 	 Total Reward: -442.5
Reward: -34.5 	 Total Reward: -426.5
Reward: -37.5 

Epoch #1:  10%|#         | 1000/10000 [00:41<06:10, 24.30it/s, env_step=1000, gradient_step=100, len=0, n/ep=0, n/st=1000, rew=0.00]

Reward: -33.5 	 Total Reward: -422.5
Reward: -1.0 	 Total Reward: -443.5
Reward: -15.5 	 Total Reward: -442.0
Reward: -11.5 	 Total Reward: -398.0
Reward: -12.5 	 Total Reward: -435.0
Reward: -10.5 	 Total Reward: -454.0
Reward: -1.0 	 Total Reward: -443.0
Reward: 8.5 	 Total Reward: -389.5
Reward: -1.0 	 Total Reward: -436.0
Reward: -2.0 	 Total Reward: -456.0
Reward: 0.0 	 Total Reward: -443.0
Reward: -11.5 	 Total Reward: -401.0
Reward: 0.0 	 Total Reward: -436.0
Reward: -1.0 	 Total Reward: -457.0
Reward: -1.0 	 Total Reward: -444.0
Reward: 0.0 	 Total Reward: -401.0
Reward: 9.5 	 Total Reward: -426.5
Reward: -1.0 	 Total Reward: -458.0
Reward: -1.0 	 Total Reward: -445.0
Reward: -10.5 	 Total Reward: -411.5
Reward: -10.5 	 Total Reward: -437.0
Reward: 0.0 	 Total Reward: -458.0
Reward: 0.0 	 Total Reward: -445.0
Reward: 0.0 	 Total Reward: -411.5
Reward: 0.0 	 Total Reward: -437.0
Reward: 1.0 	 Total Reward: -457.0
Reward: 1.0 	 Total Reward: -444.0
Reward: -10.5 	 Total Reward: -

...

Reward: 21.5 	 Total Reward: -1700.0
Reward: -124.5 	 Total Reward: -2661.5
Reward: -17.0 	 Total Reward: -2857.5
Reward: -56.5 	 Total Reward: -3520.0
Reward: 22.5 	 Total Reward: -1677.5
Reward: -96.5 	 Total Reward: -2758.0
Reward: -61.5 	 Total Reward: -2919.0
Reward: -10.0 	 Total Reward: -3530.0
Reward: 5.5 	 Total Reward: -1672.0
Reward: -23.0 	 Total Reward: -2781.0
Reward: -4.0 	 Total Reward: -2923.0
Reward: 7.5 	 Total Reward: -3522.5
Reward: 9.5 	 Total Reward: -1662.5
Reward: -55.5 	 Total Reward: -2836.5
Reward: -25.0 	 Total Reward: -2948.0
Reward: 9.5 	 Total Reward: -3513.0
Reward: -20.0 	 Total Reward: -1682.5
Reward: -1.0 	 Total Reward: -2837.5
Reward: -33.5 	 Total Reward: -2981.5
Reward: 6.5 	 Total Reward: -3506.5
Reward: 10.5 	 Total Reward: -1672.0
Reward: 8.5 	 Total Reward: -2829.0
Reward: -21.0 	 Total Reward: -3002.5
Reward: 8.5 	 Total Reward: -3498.0
Reward: 9.5 	 Total Reward: -1662.5
Reward: 0.0 	 Total Reward: -2829.0
Reward: -10.5 	 Total Reward: -301

Epoch #1:  90%|######### | 9000/10000 [18:36<02:13,  7.49it/s, env_step=9000, gradient_step=900, len=368, n/ep=4, n/st=1000, rew=-315.00]

Reward: -39.5 	 Total Reward: -1788.0
Reward: -101.5 	 Total Reward: -3123.5
Reward: -36.0 	 Total Reward: -3091.5
Reward: -158.0 	 Total Reward: -3756.0
Reward: -48.5 	 Total Reward: -1836.5
Reward: -162.0 	 Total Reward: -3285.5
Reward: -132.0 	 Total Reward: -3223.5
Reward: -149.5 	 Total Reward: -3905.5
Reward: -42.0 	 Total Reward: -1878.5
Reward: -38.0 	 Total Reward: -3323.5
Reward: -61.5 	 Total Reward: -3285.0
Reward: -95.0 	 Total Reward: -4000.5
Reward: -52.0 	 Total Reward: -1930.5
Reward: -26.5 	 Total Reward: -3350.0
Reward: 1.0 	 Total Reward: -3284.0
Reward: -22.5 	 Total Reward: -4023.0
Reward: -31.5 	 Total Reward: -1962.0
Reward: 1.5 	 Total Reward: -3348.5
Reward: -25.0 	 Total Reward: -3309.0
Reward: -27.0 	 Total Reward: -4050.0
Reward: -23.0 	 Total Reward: -1985.0
Reward: 16.0 	 Total Reward: -3332.5
Reward: -11.5 	 Total Reward: -3320.5
Reward: -5.0 	 Total Reward: -4055.0
Reward: 7.5 	 Total Reward: -1977.5
Reward: -11.5 	 Total Reward: -3344.0
Reward: -2.0 	 

Epoch #1: 10001it [19:49,  8.41it/s, env_step=10000, gradient_step=1000, len=368, n/ep=0, n/st=1000, rew=-315.00]                           


Reward: 9.5 	 Total Reward: 75.5
Reward: -10.5 	 Total Reward: 47.5
Reward: 9.5 	 Total Reward: 113.5
Reward: 9.5 	 Total Reward: 105.5
Reward: -300.0 	 Total Reward: -224.5
Reward: -224.0 	 Total Reward: -176.5
Reward: -128.5 	 Total Reward: -15.0
Reward: 138.0 	 Total Reward: 243.5
Reward: -112.5 	 Total Reward: -337.0
Reward: -294.5 	 Total Reward: -471.0
Reward: -138.5 	 Total Reward: -153.5
Reward: 189.0 	 Total Reward: 432.5
Reward: -185.0 	 Total Reward: -522.0
Reward: -131.5 	 Total Reward: -602.5
Reward: -180.5 	 Total Reward: -334.0
Reward: 109.0 	 Total Reward: 541.5
Reward: -28.5 	 Total Reward: -550.5
Reward: -36.5 	 Total Reward: -639.0
Reward: -74.5 	 Total Reward: -408.5
Reward: 88.5 	 Total Reward: 630.0
Reward: -16.0 	 Total Reward: -566.5
Reward: -42.5 	 Total Reward: -681.5
Reward: -107.5 	 Total Reward: -516.0
Reward: 43.0 	 Total Reward: 673.0
Reward: 13.0 	 Total Reward: -553.5
Reward: -21.5 	 Total Reward: -703.0
Reward: 29.0 	 Total Reward: -487.0
Reward: -49.0